In [ ]:
# ============================================================
# STEP 1 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================================
# STEP 2 — IMPORTS
# ============================================================

import os
import cv2
import json
import shutil
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
from torchvision import transforms

from sklearn.model_selection import train_test_split

# perceptual hashing
!pip install imagehash -q
import imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 16.8 MB/s eta 0:00:00


In [ ]:
# ============================================================
# STEP 1 — LOAD ORIGINAL UTKFACE DATASET
# ============================================================

import os

dataset_path = "/content/drive/MyDrive/UTKFace"

files = os.listdir(dataset_path)

print("Total Images:", len(files))
print("Sample File:", files[0])

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/UTKFace'

In [ ]:
# ============================================================
# STEP 3 — PARSE UTKFACE FILENAMES
# Format:
# age_gender_race_timestamp.jpg.chip.jpg
# ============================================================

def parse_utkface(filename):

    try:

        fname = filename.replace('.jpg.chip.jpg', '')
        fname = fname.replace('.jpg.chip', '')
        fname = fname.replace('.jpg', '')

        parts = fname.split('_')

        if len(parts) < 3:
            return None

        age    = int(parts[0])
        gender = int(parts[1])
        race   = int(parts[2])

        if age < 0 or age > 116:
            return None

        return age, gender, race

    except:
        return None

In [ ]:
# ============================================================
# STEP 4 — CREATE DATAFRAME
# ============================================================

records = []

print("Parsing filenames...")

for file in tqdm(files):

    parsed = parse_utkface(file)

    if parsed is None:
        continue

    age, gender, race = parsed

    records.append({

        'filename': file,

        'filepath': os.path.join(dataset_path, file),

        'age': age,

        'gender': gender,

        'race': race,
    })

df = pd.DataFrame(records)

print("\nTotal Parsed Images:", len(df))

print(df.head())

Parsing filenames...


100%|██████████| 23713/23713 [00:00<00:00, 438377.52it/s]


Total Parsed Images: 23713
                                filename  \
0  80_0_0_20170117204632601.jpg.chip.jpg   
1  80_0_1_20170111181750520.jpg.chip.jpg   
2  80_0_2_20170112215149951.jpg.chip.jpg   
3  80_1_0_20170110122439310.jpg.chip.jpg   
4  80_1_0_20170110140819226.jpg.chip.jpg   

                                            filepath  age  gender  race  
0  /content/drive/MyDrive/UTKFace/80_0_0_20170117...   80       0     0  
1  /content/drive/MyDrive/UTKFace/80_0_1_20170111...   80       0     1  
2  /content/drive/MyDrive/UTKFace/80_0_2_20170112...   80       0     2  
3  /content/drive/MyDrive/UTKFace/80_1_0_20170110...   80       1     0  
4  /content/drive/MyDrive/UTKFace/80_1_0_20170110...   80       1     0  


In [ ]:
# ============================================================
# STEP 5 — REMOVE CORRUPT IMAGES
# ============================================================

valid_rows = []

print("Checking corrupt images...")

for idx, row in tqdm(df.iterrows(), total=len(df)):

    try:

        img = Image.open(row['filepath'])

        img.verify()

        valid_rows.append(row)

    except:
        continue

df = pd.DataFrame(valid_rows).reset_index(drop=True)

print("\nAfter removing corrupt images:", len(df))

Checking corrupt images...


100%|██████████| 23713/23713 [2:15:14<00:00,  2.92it/s]



After removing corrupt images: 23713


In [ ]:
# ============================================================
# STEP 6 — DEFINE AGE GROUPS
# ============================================================

AGE_GROUPS = {

    0: (0, 10, '0-10'),

    1: (11, 20, '11-20'),

    2: (21, 30, '21-30'),

    3: (31, 40, '31-40'),

    4: (41, 50, '41-50'),

    5: (51, 60, '51-60'),

    6: (61, 70, '61-70'),

    7: (71, 120, '71+'),
}


def get_age_group(age):

    for idx, (low, high, label) in AGE_GROUPS.items():

        if low <= age <= high:
            return idx

    return 7

In [ ]:
# ============================================================
# STEP 7 — ADD AGE GROUP COLUMNS
# ============================================================

df['group'] = df['age'].apply(get_age_group)

df['group_label'] = df['group'].apply(
    lambda x: AGE_GROUPS[x][2]
)

print("\nAGE GROUP DISTRIBUTION")
print("=" * 50)

for idx in range(8):

    count = (df['group'] == idx).sum()

    percentage = count / len(df) * 100

    label = AGE_GROUPS[idx][2]

    print(
        f'Group {idx} ({label:6s}) : '
        f'{count:5d} ({percentage:5.1f}%)'
    )


AGE GROUP DISTRIBUTION
Group 0 (0-10  ) :  3218 ( 13.6%)
Group 1 (11-20 ) :  1659 (  7.0%)
Group 2 (21-30 ) :  7789 ( 32.8%)
Group 3 (31-40 ) :  4339 ( 18.3%)
Group 4 (41-50 ) :  2100 (  8.9%)
Group 5 (51-60 ) :  2211 (  9.3%)
Group 6 (61-70 ) :  1172 (  4.9%)
Group 7 (71+   ) :  1225 (  5.2%)


In [ ]:
# ============================================================
# STEP 8 — CREATE BALANCED DATAFRAME
# ============================================================

TARGETS = {

    0: 1500,   # 0-10

    1: 1500,   # 11-20

    2: 3000,   # 21-30

    3: 3000,   # 31-40

    4: 1800,   # 41-50

    5: 1500,   # 51-60

    6: 1200,   # 61-70

    7: 1000,   # 71+
}

balanced_parts = []

print("\nCreating balanced dataset...")

for group_id, target_size in TARGETS.items():

    group_df = df[df['group'] == group_id]

    available = len(group_df)

    print(
        f'Group {group_id} | '
        f'Available: {available}'
    )

    if available <= target_size:

        sampled = group_df

    else:

        sampled = group_df.sample(
            n=target_size,
            random_state=42
        )

    balanced_parts.append(sampled)

df_balanced = pd.concat(
    balanced_parts,
    ignore_index=True
)

df_balanced = df_balanced.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nFINAL BALANCED DATASET SIZE:", len(df_balanced))


Creating balanced dataset...
Group 0 | Available: 3218
Group 1 | Available: 1659
Group 2 | Available: 7789
Group 3 | Available: 4339
Group 4 | Available: 2100
Group 5 | Available: 2211
Group 6 | Available: 1172
Group 7 | Available: 1225

FINAL BALANCED DATASET SIZE: 14472


In [ ]:
# ============================================================
# STEP 9 — CHECK FINAL DISTRIBUTION
# ============================================================

print("\nBALANCED DISTRIBUTION")
print("=" * 50)

for idx in range(8):

    count = (df_balanced['group'] == idx).sum()

    percentage = count / len(df_balanced) * 100

    label = AGE_GROUPS[idx][2]

    print(
        f'Group {idx} ({label:6s}) : '
        f'{count:5d} ({percentage:5.1f}%)'
    )


BALANCED DISTRIBUTION
Group 0 (0-10  ) :  1500 ( 10.4%)
Group 1 (11-20 ) :  1500 ( 10.4%)
Group 2 (21-30 ) :  3000 ( 20.7%)
Group 3 (31-40 ) :  3000 ( 20.7%)
Group 4 (41-50 ) :  1800 ( 12.4%)
Group 5 (51-60 ) :  1500 ( 10.4%)
Group 6 (61-70 ) :  1172 (  8.1%)
Group 7 (71+   ) :  1000 (  6.9%)


In [ ]:
# ============================================================
# STEP 10 — TRAIN / VALIDATION SPLIT
# ============================================================

df_train, df_val = train_test_split(

    df_balanced,

    test_size=0.15,

    stratify=df_balanced['group'],

    random_state=42
)

df_train = df_train.reset_index(drop=True)

df_val = df_val.reset_index(drop=True)

print("\nTrain Size:", len(df_train))

print("Validation Size:", len(df_val))


Train Size: 12301
Validation Size: 2171


In [ ]:
# ============================================================
# STEP 11 — SAVE PREPROCESSING FILES
# ============================================================

save_path = "/content/drive/MyDrive/FaceAgingProject/preprocessing"

os.makedirs(save_path, exist_ok=True)

df_balanced.to_csv(
    f"{save_path}/df_balanced.csv",
    index=False
)

df_train.to_csv(
    f"{save_path}/df_train.csv",
    index=False
)

df_val.to_csv(
    f"{save_path}/df_val.csv",
    index=False
)

print("\nPreprocessing files saved successfully!")


Preprocessing files saved successfully!
